# Agrupamiento de circuitos por UITI acumulado y numero de eventos

Cuaderno autosuficiente y de una sola figura. Carga `data/Indicadores_vano_v3.csv`, acumula por
`CIRCUITO` el UITI (`UITI_VANO`) y el numero de eventos, y segmenta los circuitos con K-Means en
**4 grupos** (`Bajo`, `Medio`, `Medio-Alto`, `Alto`) sobre el espacio ajustado.

La figura viene con un panel de dos controles independientes:

- **Desde / Hasta**: calendarios que acotan el periodo sobre el que se acumulan UITI y eventos.
  Por defecto, el rango completo de la base.
- **Descargar etiquetas (CSV)**: baja la tabla de circuitos etiquetados con el periodo que este
  a la vista. La ultima celda hace lo mismo desde Python, con `tabla_etiquetas()` /
  `guardar_etiquetas()`: mismo esquema y mismo orden, para que los dos caminos no diverjan.

**El espacio de agrupamiento es fijo**: eje x lineal, eje y en `log10` y escalador `minmax`. Se
aplica *antes* de correr K-Means, asi que decide la particion y no solo como se dibuja. Antes se
elegia desde el panel, con dos casillas de log y un selector de preproceso; se quitaron para que
un grupo `Alto` signifique siempre lo mismo, sin depender de en que combinacion quedo el tablero
la ultima vez que alguien lo movio. Cambiar el periodo actualiza a la vez el scatter, los
contornos de membresia, las densidades marginales y el conteo de circuitos por grupo, sobre una
particion que no se mueve.

Los grupos no se nombran por el id que devuelve K-Means (que es arbitrario) sino por el **ranking
de la mediana del UITI acumulado**: `Bajo` es el de menor mediana y `Alto` el de mayor.

> **Los calendarios ajustan a mes completo.** El menor grano precomputable es el mes: un rango
> diario exacto daria 16.471 combinaciones, imposible de embeber. La alternativa seria reimplementar
> K-Means en JavaScript, y entonces la particion que ves dejaria de ser la que calcula scikit-learn.
> El panel avisa cual es el rango efectivo cada vez que cambias una fecha.

> **Por que el panel sale de una celda de codigo y no de markdown.** JupyterLab sanitiza las celdas
> de markdown: `<input>` y `<button>` estan en su lista de tags permitidos, pero `<script>` no, y
> tampoco ningun atributo `on*`. Un calendario puesto en markdown se dibujaria y quedaria muerto.
> En la salida de una celda de codigo si corre JavaScript -- es el mismo mecanismo por el que se
> dibuja Plotly. Requiere que el cuaderno este *trusted*, igual que la figura.

In [1]:
# Descomentar solo si el entorno no tiene instaladas estas dependencias.
# %pip install pandas numpy scipy scikit-learn plotly

In [ ]:
# El tablero ya no vive en estas celdas: vive en `src/chec_tableros/agrupamiento.py`,
# con pruebas propias y contra un golden que compara el HTML byte a byte. Este cuaderno
# lo LLAMA, para que haya UNA implementacion y no dos que se separan en silencio.
#
# Lo que se pierde respecto de antes: los dos tableros ya no se pintan dentro de las
# celdas. El modulo no corre en un kernel, asi que no hay nada que `display` pueda
# recibir. El documento que se abre en el navegador es el MISMO -- el tablero de VANOS,
# que es el que siempre se exporto -- y ademas usa todo el ancho de la pantalla.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from chec_tableros import agrupamiento

# ABRIR_EN_NAVEGADOR: ponlo en False para escribir el archivo sin abrir nada (Databricks,
# Colab, nbconvert). La aplicacion de escritorio no pasa por aqui: llama al modulo.
ABRIR_EN_NAVEGADOR = True

RUTA_PANEL = agrupamiento.construir(raiz=ROOT, abrir=ABRIR_EN_NAVEGADOR)
print(f'panel de vanos en {RUTA_PANEL.relative_to(ROOT)} '
      f'({RUTA_PANEL.stat().st_size / 1024 ** 2:,.1f} MB)')


## Como leerlo

- Un punto es un **circuito**, no un vano: el eje x es cuantos eventos registro en el periodo
  elegido y el eje y cuanto UITI acumulo en ese mismo periodo.
- Las **regiones sombreadas** son la particion del plano, no un contorno de densidad: marcan
  que grupo le tocaria a un circuito segun donde caiga. Son celdas de Voronoi de los cuatro
  centroides, dibujadas en el espacio ajustado. Como ese espacio es fijo, las fronteras son
  siempre las mismas: cambiar el periodo mueve los circuitos, no la particion.
- **Los titulos de las barras y de los violines dicen cuantas muestras resumen** (`n = ...`).
  Sin ese numero, dos rangos con reparto parecido se leen igual aunque uno tenga la mitad de
  circuitos con eventos. Aca el conteo va en el titulo del EJE, porque este tablero no lleva
  titulos de subplot.
- Los **violines** muestran la distribucion completa de cada variable dentro de cada grupo,
  con su caja y su mediana. Son la contraparte por grupo de los KDE marginales: aquellos
  proyectan todos los grupos sobre un mismo eje, estos los separan.
- El **diagrama de barras** cuenta cuantos circuitos quedaron en cada grupo. El reparto sale
  desbalanceado a proposito: solo el eje del UITI va en `log10`, y esa es la variable que
  ordena los nombres.
- **Solo el eje y se transforma.** Las dos variables no tienen la misma forma: el UITI
  acumulado abarca varios ordenes de magnitud y en lineal los circuitos tranquilos se apilan
  contra el cero, mientras que el numero de eventos ya se reparte de forma legible. Antes las
  dos escalas se elegian desde el panel; ahora la combinacion es una sola.
- El nombre del grupo (`Bajo` a `Alto`) sigue siendo un ranking **relativo al periodo
  elegido**. Al mover los calendarios, cambia que circuitos entran y donde caen, pero no las
  fronteras: no es una etiqueta fija del circuito, aunque ya no dependa del espacio.

**Limitacion.** `k=4` es una decision operativa (cuatro niveles de riesgo para priorizar
mantenimiento), no un valor que estos dos features impongan. Este cuaderno no valida `k`; se
limita a mostrar la particion y a nombrarla de forma consistente.

---

# Segundo tablero: agrupamiento a nivel de vano

Mismo procedimiento y mismas visualizaciones que el tablero de circuitos, pero cambiando la
unidad: aqui cada punto es un **vano** (`FID_VANO`), no un circuito. Son 27.390 vanos contra 208
circuitos, asi que todo lo anterior se repite dos ordenes de magnitud mas denso.

> **Un rango corto vuelve el agrupamiento degenerado.** Sobre el rango completo la mediana es de
> 3 eventos por vano. Sobre **un solo mes**, apenas 10.089 de los 27.390 vanos registran algun
> evento y el **55,8% de esos tiene exactamente uno**, con mediana 1. El eje de eventos se
> aplasta contra el valor 1 y K-Means termina cortando casi solo por UITI. Los rangos cortos
> siguen disponibles, pero conviene leerlos sabiendo esto; el panel avisa cuantos vanos entraron.

> **Como entra tanto dato sin inflar el cuaderno.** Replicar el esquema del primer tablero
> (21 rangos x 8 espacios de coordenadas y etiquetas) pesaria unos 20 MB. En su lugar viaja una
> matriz **vano x mes** de 1,2 MB y el navegador suma los meses del rango elegido, que para una
> suma y un conteo da exactamente lo mismo. Los grupos tampoco viajan: se derivan de los
> centroides con la regla de centroide mas cercano, la misma que dibuja los contornos y que el
> cuaderno verifica contra las etiquetas de scikit-learn.

### Como leerlo

- Un punto es un **vano**, no un circuito. El mismo vano pertenece siempre al mismo circuito,
  que aparece en el tooltip, pero el agrupamiento no lo usa: se decide solo por los eventos y el
  UITI del propio vano.
- **Los titulos de las barras y de los violines dicen cuantas muestras resumen** (`n = ...`).
  El conteo es el de vanos con eventos en el rango elegido, no los 27.390 del total.
- Los grupos **no son comparables con los del primer tablero**, aunque compartan nombre. Son dos
  particiones distintas sobre unidades distintas: un vano `Alto` vive casi siempre en un circuito
  `Alto`, pero un circuito `Alto` contiene vanos de los cuatro grupos.
- El **contorno** es la particion del plano por celdas de Voronoi de los cuatro centroides, igual
  que arriba, y en el mismo espacio fijo: eje x lineal, eje y en `log10` y `minmax`. Las curvas
  marginales son densidades por grupo calculadas en el navegador con el mismo ancho de banda de
  Scott que usa `scipy`; el cuaderno compara ambas antes de embeberlas.

- La nube sale **rayada en vertical** y la densidad marginal de arriba, con picos: no es un
  artefacto de dibujo. A nivel de vano `num_eventos` es un entero pequeno (mediana 3), asi que
  todos los vanos con el mismo conteo caen exactamente en la misma abscisa. Un KDE sobre una
  variable discreta suaviza entre valores que no existen; conviene leer esa curva como la altura
  de cada barra entera, no como una densidad continua.

**Limitacion.** Con la mediana en 3 eventos por vano sobre el rango completo, y en 1 sobre un
mes, la coordenada de eventos aporta poca informacion a nivel de vano: buena parte de la
particion termina decidida por el UITI. Es una diferencia real respecto del tablero de circuitos,
donde las dos coordenadas pesan parecido, y el espacio fijo de este cuaderno -- con `log10` solo
en el UITI -- la acentua: el eje que mas separa es justamente el transformado.